<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/exercise_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Install dependencies as needed
# pip install -U kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Correct file inside dataset
file_path = "All Muscle Training Exercises Dataset/exercise_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "nitinr00/all-muscle-training-exercises-dataset",
    file_path,
)

print("First 5 records:")
print(df.head())

/tmp/ipykernel_200/1231449284.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'all-muscle-training-exercises-dataset' dataset.
First 5 records:
                                   name     target    bodyPart  \
0            lever seated hip abduction  abductors  upper legs   
1                    side hip abduction  abductors  upper legs   
2       straight leg outer hip abductor  abductors  upper legs   
3             side bridge hip abduction  abductors  upper legs   
4  resistance band seated hip abduction  abductors  upper legs   

          equipment  
0  leverage machine  
1       body weight  
2       body weight  
3       body weight  
4   resistance band  


In [16]:
from sklearn.preprocessing import LabelEncoder

In [17]:
le_target = LabelEncoder()
le_body = LabelEncoder()
le_equipment = LabelEncoder()
le_name = LabelEncoder()

df["target"] = le_target.fit_transform(df["target"])
df["bodyPart"] = le_body.fit_transform(df["bodyPart"])
df["equipment"] = le_equipment.fit_transform(df["equipment"])
df["name"] = le_name.fit_transform(df["name"])

In [18]:
df

,name,target,bodyPart,equipment
0,144,0,8,9
1,157,0,8,4
2,164,0,8,4
3,156,0,8,4
4,151,0,8,11
...,...,...,...,...
163,102,18,0,5
164,103,18,0,5
165,106,18,0,5
166,107,18,0,5


In [19]:
import torch

X = df[["target","bodyPart","equipment"]].values
y = df["name"].values

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [20]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(3, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, len(df["name"].unique()))
)

In [21]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(200):

    outputs = model(X)
    loss = loss_fn(outputs, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(epoch, loss.item())

0 5.445893287658691
20 4.7882819175720215
40 4.214544773101807
60 3.65392804145813
80 3.2233433723449707
100 2.8939554691314697
120 2.6291706562042236
140 2.4118168354034424
160 2.23545503616333
180 2.093514919281006


In [23]:
sample = torch.tensor([[1,2,3]], dtype=torch.float32)

pred = model(sample)
predicted = torch.argmax(pred).item()

print(le_name.inverse_transform([predicted]))

['dumbbell single leg calf raise']
